# Clark tuning analyses - presentation figures

Curated, configurable figures for the example **animal 0, session 0**.
Edit the **global configuration** cell for shared visual parameters and the
per-figure **config** cells for figure-specific parameters. Two global flags
control behaviour:

- `COMPUTE_MODE`: `'fresh'` recomputes all fits and writes the cache; `'cache'`
  loads the pre-computed cache (fast).
- `OUTPUT_MODE`: `'save'` writes PNGs to `FIG_DIR`; `'display'` shows them inline.

In [1]:
import os, numpy as np, polars as pl
import matplotlib.pyplot as plt
import clark_lib as L           # compute functions (see clark_lib.py)

ROOT = "/ceph/branco/Jake/training_data_barrier"
DT = 0.025
BEHAVE = ['hdir', 'bdir', 'hsa', 'hba']        # behavioural variables (order = index)
HDIR, BDIR, HSA, HBA = 0, 1, 2, 3
ALL_SESSIONS = {0: [0, 1], 1: [0, 1, 2, 3, 4], 2: [0, 1, 2, 3, 4]}
EX_ANIMAL, EX_SESSION = 0, 0
EX_ANIMAL_NAME, EX_SESSION_NAME = 'JAL006', '2024_03_25T11_05_33'

## Global configuration (shared visual parameters + flags)

In [33]:
# ===== GLOBAL FLAGS =====
COMPUTE_MODE = 'cache'      # 'fresh' (recompute + write cache) | 'cache' (load)
OUTPUT_MODE  = 'save'    # 'save' (PNG -> FIG_DIR) | 'display' (inline)
CACHE_DIR = 'presentation_cache'
FIG_DIR   = 'presentation_figures'
FIG_DPI   = 120

# ===== RESOLUTIONS =====
D1     = 100      # 1-D tuning resolution (items 2,3,7,8,9,10,11)
D2     = 20       # 2-D tuning resolution (items 4,5)
D1_REG = 100      # item-6 1-D regression bins   (matched: D1_REG = D2_REG**2)
D2_REG = 10       # item-6 2-D regression bins/axis

# ===== REGRESSION TRAIN/TEST SPLIT (held-out variance explained; items 6 & 10) =====
TEST_FRAC  = 0.5    # fraction of timepoints held out for evaluation (0 = in-sample)
SPLIT_SEED = 0      # RNG seed for the train/test timepoint split

# ===== DISPLAY NAMES =====
VAR_NAMES = {
    'hdir': 'Head direction', 'bdir': 'Barrier direction',
    'hsa': 'Head-shelter angle', 'hba': 'Head-barrier angle',
}

# ===== FONT SIZES (by label type) =====
FONT = {'suptitle': 24, 'title': 22, 'axis_label': 18, 'tick': 16,
        'legend': 16, 'annotation': 14}

# ===== COLOUR SCHEME (by data type) =====
COLORS = {
    'behave': {'hdir': '#1f77b4', 'bdir': '#d62728', 'hsa': '#2ca02c', 'hba': '#9467bd',
               'position': "#bcc8d1"},
    'data':   {'raw': '0.6', 'smoothed': '#1f77b4', 'mean': 'k', 'std': '0.4',
               'generative': '#ff7f0e', 'real': 'k',
               'partition_false': '#1f77b4', 'partition_true': '#d62728'},
    'animals': ['#1b9e77', '#d95f02', '#7570b3'],
}
CMAP_2D   = 'viridis'
CMAP_CORR = 'magma'

plt.rcParams.update({
    'figure.titlesize': FONT['suptitle'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis_label'], 'xtick.labelsize': FONT['tick'],
    'ytick.labelsize': FONT['tick'], 'legend.fontsize': FONT['legend'],
})

## Compute / cache layer

In [3]:
def load_session_arrays(a, s, barrier=False, B_select=BEHAVE):
    sp = os.path.join(ROOT, str(a), str(s))
    df = pl.scan_csv(os.path.join(sp, 'rates.csv')).collect()
    nc = [c for c in df.columns if c.isdigit()]
    out = dict(X=df.select(nc).to_numpy().T, B=df.select(B_select).to_numpy().T,
               S=pl.scan_csv(os.path.join(sp, 'spikes.csv')).collect().select(nc).to_numpy().T,
               neuron_cols=nc)
    if barrier:
        out['barrier_flipped'] = df.select('barrier_flipped').to_numpy().squeeze()
    return out

def fit_session_1d(X, B, S):
    return L.gp_cross_validated_tuning_curves(
        X, B, S, D=D1, K=2, L=2,
        length_scales=np.logspace(-2, 0, 15), amplitudes=np.logspace(-2, 0, 15),
        sigma=1, gram_fns=L.periodic_gaussian_gram, gram_fn_args={"period": 2 * np.pi},
        length_scale_key="length_scale", dt=DT, equalise_histogram=True)

def fit_session_2d_bh(X, B, S):
    r = L.gp_cross_validated_tuning_curves_2d(
        X, B, S, D=D2, K=2, L=2,
        length_scales=np.logspace(-2, 0, 8), amplitudes=np.logspace(-2, 0, 8),
        sigma=1, gram_fn_args={"period": 2 * np.pi}, dt=DT, pairs=[(BDIR, HBA)])
    return r['F_best_pair'][:, 0], r['scores_pair'][:, 0]

def bin_behaviour(B, D):
    P, T = B.shape
    bins = np.full((P, T), -1, int); doms = []
    for p in range(P):
        b, dom = L._bin_behaviour_row_and_domain(B[p], D)
        bins[p] = b
        doms.append(dom)
    return bins, doms

def compute_or_load():
    sessions = [(a, s) for a, ss in ALL_SESSIONS.items() for s in ss]
    RES = dict(sessions=sessions, fit1d={}, fit2d={})
    if COMPUTE_MODE == 'cache':
        g = np.load(f"{CACHE_DIR}/global.npz")
        RES['N_global'], RES['T'] = int(g['N_global']), float(g['T'])
        for a, s in sessions:
            RES['fit1d'][(a, s)] = dict(np.load(f"{CACHE_DIR}/fit1d_{a}_{s}.npz"))
            RES['fit2d'][(a, s)] = dict(np.load(f"{CACHE_DIR}/fit2d_{a}_{s}.npz"))
        gp = np.load(f"{CACHE_DIR}/gen1d_ex.npz")
        RES['gen'] = dict(sigma=gp['sigma'], beta=gp['beta'], b=gp['b'])
        RES['barrier'] = dict(np.load(f"{CACHE_DIR}/barrier_ex.npz"))
        return RES
    # ---- fresh compute ----
    os.makedirs(CACHE_DIR, exist_ok=True)
    N_global = 0
    for a, s in sessions:
        d = load_session_arrays(a, s)
        f = fit_session_1d(d['X'], d['B'], d['S'])
        RES['fit1d'][(a, s)] = dict(
            F_raw = f['F_raw'],
            F_smooth = f['F_smooth'],
            F_best=f['F_best'], 
            F_best_normed=f['F_best_normed'], 
            scores=f['scores'])
        np.savez(f"{CACHE_DIR}/fit1d_{a}_{s}.npz", **RES['fit1d'][(a, s)])
        N_global += f['F_best'].shape[0]
        del d
    RES['N_global'] = N_global
    RES['T'] = L.significance_threshold(N_global)
    np.savez(f"{CACHE_DIR}/global.npz", N_global=N_global, T=RES['T'])
    for a, s in sessions:
        d = load_session_arrays(a, s)
        Fp, sp = fit_session_2d_bh(d['X'], d['B'], d['S'])
        RES['fit2d'][(a, s)] = dict(F_pair=Fp, scores_pair=sp)
        np.savez(f"{CACHE_DIR}/fit2d_{a}_{s}.npz", **RES['fit2d'][(a, s)])
        del d
    ex = RES['fit1d'][(EX_ANIMAL, EX_SESSION)]
    mask = L.significance_mask(ex['scores'], RES['T'])
    gen = L.fit_sampled_tuning_curve_correlation_model(
        F=ex['F_best_normed'][mask], sigma_values=np.linspace(0.5, 3.0, 25),
        beta_values=np.linspace(1e-2, 10.0, 25), b_values=np.linspace(0.0, 4.0, 25),
        n_samples=1000, fit="per_p", seed=0, jitter=1e-6)['best_per_p_params']
    RES['gen'] = dict(sigma=gen['sigma'], beta=gen['beta'], b=gen['b'])
    np.savez(f"{CACHE_DIR}/gen1d_ex.npz", **RES['gen'])
    db = load_session_arrays(EX_ANIMAL, EX_SESSION, barrier=True)
    m = db['barrier_flipped'].astype(bool)
    fF = fit_session_1d(db['X'][:, m], db['B'][:, m], db['S'][:, m])
    fT = fit_session_1d(db['X'][:, ~m], db['B'][:, ~m], db['S'][:, ~m])
    RES['barrier'] = dict(FbF=fF['F_best_normed'], FbT=fT['F_best_normed'],
                          scF=fF['scores'], scT=fT['scores'])
    np.savez(f"{CACHE_DIR}/barrier_ex.npz", **RES['barrier'])
    return RES

RES = compute_or_load()
print(f"N_global={RES['N_global']}  significance threshold T={RES['T']:.3f}")

N_global=2075  significance threshold T=4.618


In [4]:
# Shared plotting / pooling helpers.
ANGLE = np.linspace(-np.pi, np.pi, D1, endpoint=False)

def finalize(fig, name):
    if OUTPUT_MODE == 'save':
        os.makedirs(FIG_DIR, exist_ok=True)
        path = os.path.join(FIG_DIR, name + '.png')
        fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight'); plt.close(fig)
        print('saved', path)
    else:
        plt.show()

def filtered_pool_1d():
    """Pool significance-filtered 1-D curves across all sessions.
    Returns curves (Npool,P,D1) and animal id per neuron."""
    curves, animal = [], []
    for (a, s) in RES['sessions']:
        f = RES['fit1d'][(a, s)]
        m = L.significance_mask(f['scores'], RES['T'])
        curves.append(f['F_best_normed'][m])
        animal.append(np.full(int(m.sum()), a))
    return np.concatenate(curves, 0), np.concatenate(animal, 0)

def filtered_pool_2d():
    """Pool significance-filtered bdir-hba 2-D maps across all sessions."""
    maps = []
    for (a, s) in RES['sessions']:
        f1 = RES['fit1d'][(a, s)]
        m = L.significance_mask(f1['scores'], RES['T'])
        maps.append(RES['fit2d'][(a, s)]['F_pair'][m])
    return np.concatenate(maps, 0)

def unwrap(B, eps=np.pi):
    y = []
    for i, b in enumerate(B):
        if i > 0 and np.abs(b - y[i-1]) > eps:
            y.extend([np.nan, b])
        else:
            y.append(b)
    return np.array(y)

## 0. Behavioural video

In [ ]:
ITEM0 = dict(
    window=(0, 1000),                 # (start_frame, n_frames)
    figsize=(12, 12),
)

In [ ]:
d = load_session_arrays(EX_ANIMAL, EX_SESSION, B_select=BEHAVE + ['frame', 'x', 'y', 'shelter_dist', 'barrier_dist'])
t0, n = ITEM0['window']
frames = d['B'][4].astype(int)
f0     = frames[t0]
fig, ax = plt.subplots(1, 1, figsize=ITEM0['figsize'])

vp = '/ceph/branco/Jasmine_Laurence/Experimental_Data/JAL006/JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33/JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33_cam.avi'

import cv2
import imageio.v2 as imageio
from tqdm.auto import tqdm

if OUTPUT_MODE == 'save':
    os.makedirs(FIG_DIR, exist_ok=True)
    out_vp = os.path.join(
        FIG_DIR,
        f"{EX_SESSION}_cam_overlay.mp4",
    )
else:
    out_vp = os.path.join(
        CACHE_DIR,
        f"{EX_SESSION}_cam_overlay.mp4",
    )

cap = cv2.VideoCapture(vp)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {vp}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps <= 0 or np.isnan(fps):
    fps = 30

# Seek once, then read sequentially frame-by-frame
cap.set(cv2.CAP_PROP_POS_FRAMES, f0)

with imageio.get_writer(
    out_vp, 
    fps=fps, 
    codec="libopenh264",
    output_params=[
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
    ]
) as writer:
    for local_i, frame_i in enumerate(tqdm(range(f0, f0 + n))):
        ok, frame = cap.read()
        if not ok:
            print(f"Stopped early: could not read frame {frame_i}")
            break

        # OpenCV gives BGR; matplotlib expects RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        ax.clear()
        ax.imshow(frame, aspect='equal')
        ax.set_axis_off()

        idx, = np.where(frames == frame_i)
        if len(idx) == 0:
            continue
        else:
            idx = idx[0]

        B_i = np.asarray(d["B"][:, idx]) # (9,)

        hdir, bdir, hsa, hba, _, x, y, sd, bd = B_i
        sdir = hdir + hsa
        scale = 0.9
        x, y = 1024-x, 1024-y
        l = 100

        ax.scatter(x, y, color=COLORS['behave']['position'])
        ax.plot([x, x-l*np.cos(hdir)], [y, y-l*np.sin(hdir)], color=COLORS['behave']['hdir'])
        ax.plot([x, x-l*np.cos(bdir)], [y, y-l*np.sin(bdir)], color=COLORS['behave']['bdir'], zorder=10)
        ax.plot([x, x-bd*np.cos(bdir)], [y, y-bd*np.sin(bdir)], color=COLORS['behave']['bdir'], alpha=0.5, ls='--', zorder=0)
        ax.plot([x, x-l*np.cos(sdir)], [y, y-l*np.sin(sdir)], color=COLORS['behave']['hsa'], zorder=0)
        ax.plot([x, x-sd*np.cos(sdir)], [y, y-sd*np.sin(sdir)], color=COLORS['behave']['hsa'], alpha=0.5, ls='--', zorder=0)


        ax.set_xlim(0, frame.shape[1])
        ax.set_ylim(frame.shape[0], 0)

        fig.canvas.draw()

        # Convert rendered matplotlib canvas to RGB video frame
        rendered = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
        writer.append_data(rendered)

cap.release()
plt.close(fig)

print(f"Saved overlay video to: {out_vp}")


## 1. Example behaviour & neuron time series

In [34]:
ITEM1 = dict(
    window=(0, 1000),                 # (start_frame, n_frames)
    cells={'hdir': 108},   # first cell per variable
    figsize=(12, 10), lw=1.2,
    spike_height=2.5
)

In [35]:
d = load_session_arrays(EX_ANIMAL, EX_SESSION)
t0, n = ITEM1['window']
sl = slice(t0, t0 + n)
sh = ITEM1['spike_height']
t = np.arange(n) * DT
fig, axes = plt.subplots(1 + len(ITEM1['cells']), 1, figsize=ITEM1['figsize'], sharex=True)
for p, v in enumerate(BEHAVE):
    axes[0].plot(t, d['B'][p, sl], color=COLORS['behave'][v], lw=ITEM1['lw'],
                 label=VAR_NAMES[v], alpha=0.8)
axes[0].set_title('Behavioural variables'); axes[0].set_ylabel('value (rad)')
axes[0].legend(ncol=1, loc='upper right')
axes[0].set_yticks([-np.pi/2, 0, np.pi/2])
axes[0].set_yticklabels([r'$-\pi / 2$', r'$0$', r'$\pi / 2$'])
for p, v in enumerate(BEHAVE):
    if v not in ITEM1['cells']:
        continue
    nidx = ITEM1['cells'][v]; ax = axes[1 + p]
    ax.plot(t, d['X'][nidx, sl], color=COLORS['behave'][v], lw=ITEM1['lw'], zorder=10)
    ax.plot(t, sh*d['S'][nidx, sl], color=COLORS['behave'][v], lw=ITEM1['lw']*0.5, zorder=0, alpha=0.5)
    ax.set_ylabel('rate (Hz)'); ax.set_title(f'Neuron #{nidx}')
    ax.set_ylim(0, sh)
axes[-1].set_xlabel('time (s)')
fig.suptitle(f'Example data (animal: {EX_ANIMAL_NAME}, session: {EX_SESSION_NAME})')
fig.tight_layout(); finalize(fig, 'item1_timeseries')

saved presentation_figures/item1_timeseries.png


## 2. Example 1-D tuning curves (selected cells)

In [36]:
ITEM2 = dict(
    cells={'hdir': [108, 54, 2, 64], 'bdir': [79, 96, 108, 50],
           'hsa': [73, 108, 75, 58], 'hba': [71, 57, 115, 108]},
    figsize=(16, 13), lw=2.0,
    ylabel='Normalized firing rate',
    legend_labels={
        'raw':  'raw partition tuning curve',
        'cv':   'best cross-validated tuning curve (per partition)',
        'best': 'overall best tuning curve',
    },
)

In [37]:
from matplotlib.lines import Line2D

F_raw    = RES['fit1d'][(EX_ANIMAL, EX_SESSION)]['F_raw'].copy()
F_raw   /= F_raw.mean(axis=2, keepdims=True)
F_smooth = RES['fit1d'][(EX_ANIMAL, EX_SESSION)]['F_smooth'].copy()
F_smooth/= F_smooth.mean(axis=2, keepdims=True)
F_best   = RES['fit1d'][(EX_ANIMAL, EX_SESSION)]['F_best_normed']
ncol = 4

fig = plt.figure(figsize=ITEM2['figsize'], layout='constrained')
# one subfigure per behavioural variable (label above the row) + a thin row for the legend
row_sfs = fig.subfigures(len(BEHAVE) + 1, 1, height_ratios=[1] * len(BEHAVE) + [0.18])
for p, v in enumerate(BEHAVE):
    sf = row_sfs[p]
    sf.suptitle(VAR_NAMES[v], fontsize=FONT['title'], fontweight='bold')
    axes = sf.subplots(1, ncol, sharex=True)
    for j, nidx in enumerate(ITEM2['cells'][v]):
        ax = axes[j]
        for k in range(F_raw.shape[-1]):
            a = 0.5 if k == 0 else 1.0
            ax.plot(ANGLE, F_raw[nidx, p, :, k], color=COLORS['behave'][v],
                    lw=ITEM2['lw'] * 0.5, alpha=a, ls='--')
            ax.plot(ANGLE, F_smooth[nidx, p, :, k], color=COLORS['behave'][v],
                    lw=ITEM2['lw'] * 0.5, alpha=a, ls=':')
        ax.plot(ANGLE, F_best[nidx, p], color=COLORS['behave'][v], lw=ITEM2['lw'])
        ax.set_title(f'#{nidx}', fontsize=FONT['title'])
        ax.set_xlim(-np.pi, np.pi)
        ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
        if j == 0:
            ax.set_ylabel(ITEM2['ylabel'])

# shared legend below the whole figure: one entry per line type
handles = [Line2D([], [], color='k', ls='--', label=ITEM2['legend_labels']['raw']),
           Line2D([], [], color='k', ls=':',  label=ITEM2['legend_labels']['cv']),
           Line2D([], [], color='k', ls='-',  label=ITEM2['legend_labels']['best'])]
row_sfs[-1].legend(handles=handles, loc='center', ncol=3, frameon=False,
                   fontsize=FONT['legend'])
fig.suptitle(f'Example 1-D tuning (animal: {EX_ANIMAL_NAME}, session: {EX_SESSION_NAME})')
finalize(fig, 'item2_tuning')

saved presentation_figures/item2_tuning.png


## 3. Pooled COM-aligned 1-D tuning: mean & std

Filtered cells pooled across all animals/sessions, circularly aligned by centre
of mass; global mean/std with per-animal mean/std overlaid.

In [38]:
ITEM3 = dict(figsize=(16, 4), lw_global=2.5, lw_animal=1.5, alpha_animal=0.8)

In [39]:
pool, animal_id = filtered_pool_1d()
aligned = {p: L.align_by_com_1d(pool[:, p, :]) for p in range(len(BEHAVE))}

for stat, fn in [('mean', np.nanmean), ('std', np.nanstd)]:
    fig, axes = plt.subplots(1, len(BEHAVE), figsize=ITEM3['figsize'])
    for p, v in enumerate(BEHAVE):
        ax = axes[p]
        A = aligned[p]
        ax.plot(ANGLE, fn(A, axis=0), color=COLORS['data'][stat],
                lw=ITEM3['lw_global'], label='global', zorder=3)
        for ai, a in enumerate(ALL_SESSIONS):
            sel = animal_id == a
            if sel.sum() == 0:
                continue
            ax.plot(ANGLE, fn(A[sel], axis=0), color=COLORS['animals'][ai],
                    lw=ITEM3['lw_animal'], alpha=ITEM3['alpha_animal'], label=f'animal {a}')
        ax.set_title(VAR_NAMES[v]); ax.set_xlim(-np.pi, np.pi)
        ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
        ax.set_ylim(0, 2)
        if p == 0:
            ax.set_ylabel(f'{stat} tuning')
    fig.suptitle(f'Pooled COM-aligned 1-D tuning: {stat} (N={pool.shape[0]} cells)')
    handles = [Line2D([], [], color=COLORS['data'][stat], ls='-', label='global', lw=ITEM3['lw_global'])]
    for ai, a in enumerate(ALL_SESSIONS):
        handles.append(
            Line2D([], [], color=COLORS['animals'][ai], ls='-', label=f'animal {a}', lw=ITEM3['lw_animal'])
        )
    fig.legend(handles=handles, loc='lower center', ncol=4, frameon=True, fontsize=FONT['legend'])
    fig.tight_layout(); finalize(fig, f'item3_pooled_{stat}')

saved presentation_figures/item3_pooled_mean.png
saved presentation_figures/item3_pooled_std.png


## 4. Example 2-D tuning (barrier direction x head-barrier angle)

In [40]:
ITEM4 = dict(cells=[108, 65, 77, 72], figsize=(16, 8), cmap=CMAP_2D)

In [41]:
Fp = RES['fit2d'][(EX_ANIMAL, EX_SESSION)]['F_pair']
ext = [-np.pi, np.pi, -np.pi, np.pi]
fig = plt.figure(figsize=ITEM4['figsize'])
gs  = fig.add_gridspec(nrows=2, ncols=4, height_ratios=[1,0.1])
from matplotlib.colors import Normalize
norm = Normalize(vmin=0, vmax=0.7)
for i, nidx in enumerate(ITEM4['cells']):
    ax = fig.add_subplot(gs[0,i])
    im = ax.imshow(Fp[nidx], origin='lower', extent=ext, aspect='equal', cmap=ITEM4['cmap'], norm=norm)
    ax.set_title(f'#{nidx}')
    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_yticks([-np.pi, 0, np.pi])
    ax.set_xticklabels([r'$-\pi$', r'$0$', r'$\pi$'])
    ax.set_yticklabels([r'$-\pi$', r'$0$', r'$\pi$'])
    ax.set_xlabel(VAR_NAMES['bdir'])
    ax.set_ylabel(VAR_NAMES['hba'])
cax = fig.add_subplot(gs[1,1:3])
fig.colorbar(im, cax=cax, fraction=0.046, label='Average firing rate', orientation='horizontal')
fig.suptitle(f'Example 2-D tuning: Barrier Direction x Head-Barrier Angle (animal {EX_ANIMAL_NAME}, session {EX_SESSION_NAME})')
fig.tight_layout(); finalize(fig, 'item4_tuning2d')

saved presentation_figures/item4_tuning2d.png


## 5. Pooled 2-D tuning (bdir x hba): mean & std

In [42]:
ITEM5 = dict(figsize=(11, 4.5), cmap=CMAP_2D)

In [43]:
maps = filtered_pool_2d()
aligned2d = L.align_by_com_2d(maps)
ext = [-np.pi, np.pi, -np.pi, np.pi]
fig, axes = plt.subplots(1, 2, figsize=ITEM5['figsize'])
from matplotlib.colors import Normalize
norm = Normalize(vmin=0, vmax=0.25)
for ax, (stat, M) in zip(axes, [('mean', np.nanmean(aligned2d, 0)),
                                 ('std', np.nanstd(aligned2d, 0))]):
    im = ax.imshow(M, origin='lower', extent=ext, aspect='equal', cmap=ITEM5['cmap'], norm=norm)
    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_yticks([-np.pi, 0, np.pi])
    ax.set_xticklabels([r'$-\pi$', r'$0$', r'$\pi$'])
    ax.set_yticklabels([r'$-\pi$', r'$0$', r'$\pi$'])
    ax.set_title(f'{stat}'); ax.set_xlabel(VAR_NAMES['bdir']); ax.set_ylabel(VAR_NAMES['hba'])
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f'Pooled COM-aligned 2-D tuning: Barrier Direction x Head-Barrier Angle (N={maps.shape[0]} cells)')
fig.tight_layout(); finalize(fig, 'item5_pooled2d')

saved presentation_figures/item5_pooled2d.png


## 6. Variance explained by regressor sets (per session)

One-hot binned designs at matched per-profile parameter count
(`D1_REG = D2_REG**2`): each 1-D / 2-D profile alone, all 1-D, all 2-D, and all
together. VE over significance-filtered cells.

In [44]:
ITEM6 = dict(max_T=40000, l2=1e-2, figsize=(10, 6), seed=0,
             configs=['1D each (mean)', '1D all', '2D each (mean)', '2D all', 'all'])

In [45]:
from itertools import combinations
pairs = list(combinations(range(len(BEHAVE)), 2))
rng = np.random.default_rng(ITEM6['seed'])
fig, ax = plt.subplots(figsize=ITEM6['figsize'])
for (a, s) in RES['sessions']:
    d = load_session_arrays(a, s)
    T = d['X'].shape[1]
    sub = np.sort(rng.choice(T, size=min(ITEM6['max_T'], T), replace=False))
    Xf = d['X'][:, sub]
    b1, _ = bin_behaviour(d['B'][:, sub], D1_REG)
    b2, _ = bin_behaviour(d['B'][:, sub], D2_REG)
    oh1 = [L.onehot_bin_design(b1[p], D1_REG).T for p in range(len(BEHAVE))]       # (D,Tsub)
    ohj = [L.onehot_joint_design(b2[p], b2[q], D2_REG).T for (p, q) in pairs]
    ve_1d_each = [L.ridge_ve_shared_basis(Xf, oh1[p], ITEM6['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0] for p in range(len(BEHAVE))]
    ve_2d_each = [L.ridge_ve_shared_basis(Xf, ohj[j], ITEM6['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0] for j in range(len(pairs))]
    ve_1d_all = L.ridge_ve_shared_basis(Xf, np.vstack(oh1), ITEM6['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0]
    ve_2d_all = L.ridge_ve_shared_basis(Xf, np.vstack(ohj), ITEM6['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0]
    ve_all = L.ridge_ve_shared_basis(Xf, np.vstack(oh1 + ohj), ITEM6['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0]
    seq = [np.mean(ve_1d_each), ve_1d_all, np.mean(ve_2d_each), ve_2d_all, ve_all]
    ax.plot(range(len(seq)), seq, '-o', color=COLORS['animals'][list(ALL_SESSIONS).index(a)],
            alpha=0.7, label=f'animal {a}' if s == ALL_SESSIONS[a][0] else None)
    print(f"{a}.{s}: 1D={ve_1d_all:.3f} 2D={ve_2d_all:.3f} all={ve_all:.3f}")
    del d
ax.set_xticks(range(len(ITEM6['configs']))); ax.set_xticklabels(ITEM6['configs'])
ax.set_ylabel('variance explained'); ax.legend()
ax.set_title(f'Variance explained by tuning curves')
fig.tight_layout(); finalize(fig, 'item6_regression_ve')

0.0: 1D=0.207 2D=0.264 all=0.290
0.1: 1D=0.249 2D=0.303 all=0.327
1.0: 1D=0.186 2D=0.276 all=0.299
1.1: 1D=0.201 2D=0.269 all=0.286
1.2: 1D=0.186 2D=0.256 all=0.277
1.3: 1D=0.203 2D=0.273 all=0.292
1.4: 1D=0.175 2D=0.234 all=0.255
2.0: 1D=0.185 2D=0.246 all=0.267
2.1: 1D=0.186 2D=0.242 all=0.260
2.2: 1D=0.194 2D=0.254 all=0.285
2.3: 1D=0.201 2D=0.252 all=0.276
2.4: 1D=0.226 2D=0.282 all=0.308
saved presentation_figures/item6_regression_ve.png


## 7. Tuning-curve correlation matrices (pooled, per variable)

In [46]:
ITEM7 = dict(figsize=(16, 7), cmap=CMAP_CORR, n_svals=10)

In [47]:
pool, _ = filtered_pool_1d()
nsv = ITEM7['n_svals']
fig, axes = plt.subplots(2, len(BEHAVE), figsize=ITEM7['figsize'],
                         gridspec_kw={'height_ratios': [3, 2]})
for p, v in enumerate(BEHAVE):
    C = L.tuning_correlation(pool[:, p, :])
    im = axes[0, p].imshow(C, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
                           cmap=ITEM7['cmap'], vmin=0.5, vmax=1.5)
    axes[0, p].set_title(VAR_NAMES[v]); fig.colorbar(im, ax=axes[0, p], fraction=0.046)
    s = L.mean_subtracted_svals(C, nsv)
    s = np.square(s) / np.sum(np.square(s))
    axes[1, p].plot(np.arange(1, len(s) + 1), s, '-o', ms=4, color=COLORS['behave'][v])
    axes[1, p].set_xlabel('principal component')
    if p == 0:
        axes[1, p].set_ylabel('prop. variance explained')
fig.suptitle(f'Pooled tuning correlation matrices and spectra (N={pool.shape[0]} cells)')
fig.tight_layout(); finalize(fig, 'item7_corr')

saved presentation_figures/item7_corr.png


## 8. Draws from the fitted 1-D generative process

In [48]:
ITEM8 = dict(n_draws=5, figsize=(16, 12), lw=2.0, alpha=0.7, seed=1)

In [49]:
gp = RES['gen']
rng = np.random.default_rng(ITEM8['seed'])
draws = L.sample_basis_per_p(gp['sigma'], gp['beta'], gp['b'], ITEM8['n_draws'], D1, rng)  # (S,P,D1)
fig, axes = plt.subplots(2, 2, figsize=ITEM8['figsize'], sharey=True)
axes = axes.flatten()
for p, v in enumerate(BEHAVE):
    for s_ in range(ITEM8['n_draws']):
        axes[p].plot(ANGLE, draws[s_, p], color=f'C{s_}',
                     lw=ITEM8['lw'], alpha=ITEM8['alpha'])
    axes[p].set_title(f"{VAR_NAMES[v]}\n($\\sigma$={gp['sigma'][p]:.2f}, "
                      f"$\\beta$={gp['beta'][p]:.2f}, b={gp['b'][p]:.2f})")
    axes[p].set_xlim(-np.pi, np.pi)
    axes[p].set_xticks([-np.pi, 0, np.pi])
    axes[p].set_xticklabels([r'$-\pi$', r'$0$', r'$\pi$'])
fig.suptitle('Sample tuning curves from generative process')
fig.tight_layout(); finalize(fig, 'item8_generative_draws')

saved presentation_figures/item8_generative_draws.png


## 9. Generative correlation matrices vs sample size

In [50]:
ITEM9 = dict(sizes=[10, 50, 200, 1000], figsize=(16, 16), cmap=CMAP_CORR, seed=2)

In [51]:
gp = RES['gen']; sizes = ITEM9['sizes']
fig, axes = plt.subplots(len(BEHAVE), len(sizes), figsize=ITEM9['figsize'])
for j, Nd in enumerate(sizes):
    rng = np.random.default_rng(ITEM9['seed'])
    Q = L.sample_basis_per_p(gp['sigma'], gp['beta'], gp['b'], Nd, D1, rng)   # (Nd,P,D1)
    for p, v in enumerate(BEHAVE):
        C = L.tuning_correlation(Q[:, p, :])
        axes[p, j].imshow(C, origin='lower', cmap=ITEM9['cmap'])
        axes[p, j].set_xticks([]); axes[p, j].set_yticks([])
        if p == 0:
            axes[p, j].set_title(f'N = {Nd}')
        if j == 0:
            axes[p, j].set_ylabel(VAR_NAMES[v])
fig.suptitle('Generative process correlation matrices for increasing sample size')
fig.tight_layout(); finalize(fig, 'item9_corr_vs_samples')

saved presentation_figures/item9_corr_vs_samples.png


## 9b. Singular values of generative correlation matrices vs sample size

In [52]:
ITEM9SV = dict(sizes=[10, 50, 200, 1000], n_svals=10, figsize=(16, 12), seed=2)

In [53]:
gp = RES['gen']; sizes = ITEM9SV['sizes']; nsv = ITEM9SV['n_svals']
size_colors = plt.cm.viridis(np.linspace(0, 1, len(sizes)))
fig, axes = plt.subplots(2, 2, figsize=ITEM9SV['figsize'], sharey=True)
axes = axes.flatten()
for p, v in enumerate(BEHAVE):
    for k, Nd in enumerate(sizes):
        rng = np.random.default_rng(ITEM9SV['seed'])
        Q = L.sample_basis_per_p(gp['sigma'], gp['beta'], gp['b'], Nd, D1, rng)
        C = L.tuning_correlation(Q[:, p, :])
        s = L.mean_subtracted_svals(C, nsv)
        s = np.square(s) / np.sum(np.square(s))
        axes[p].plot(np.arange(1, len(s) + 1), s, '-o', ms=3, color=size_colors[k],
                     label=f'N = {Nd}' if p == 0 else None)
    axes[p].set_title(VAR_NAMES[v]); axes[p].set_xlabel('principal component')
    if p == 0:
        axes[p].set_ylabel('prop. variance explained'); axes[p].legend(title='sample size')
fig.suptitle('Spectra of sampled correlation matrices for increasing sample size')
fig.tight_layout(); finalize(fig, 'item9b_corr_svals')

saved presentation_figures/item9b_corr_svals.png


## 10. Variance explained by generative draws vs sample size

In [54]:
ITEM10 = dict(sizes=[1, 2, 5, 10, 20, 50, 100, 250, 1000], seeds=[0, 1, 2],
              max_T=40000, l2=1e-2, figsize=(8, 5))

In [55]:
gp = RES['gen']
d = load_session_arrays(EX_ANIMAL, EX_SESSION)
m = L.significance_mask(RES['fit1d'][(EX_ANIMAL, EX_SESSION)]['scores'], RES['T'])
rng0 = np.random.default_rng(0); T = d['X'].shape[1]
sub = np.sort(rng0.choice(T, size=min(ITEM10['max_T'], T), replace=False))
Xf = d['X'][:, sub]
bins, _ = bin_behaviour(d['B'][:, sub], D1)
Xf = d['X'][:, sub]
oh1 = [L.onehot_bin_design(bins[p], D1).T for p in range(len(BEHAVE))]       # (D,Tsub)
ve_real = L.ridge_ve_shared_basis(Xf, np.vstack(oh1), ITEM10['l2'], test_frac=TEST_FRAC, split_seed=SPLIT_SEED)[0]
    
P = len(BEHAVE); sizes = ITEM10['sizes']
VE = np.zeros((len(ITEM10['seeds']), len(sizes)))
for si, seed in enumerate(ITEM10['seeds']):
    rng = np.random.default_rng(100 + seed)
    for j, Sn in enumerate(sizes):
        Q = L.sample_basis_per_p(gp['sigma'], gp['beta'], gp['b'], Sn, D1, rng)
        design = L.gather_tuning_basis(Q, bins).reshape(Sn * P, -1)
        VE[si, j], _ = L.ridge_ve_shared_basis(Xf, design, ITEM10['l2'],
                                               test_frac=TEST_FRAC, split_seed=SPLIT_SEED)
mean, sem = VE.mean(0), VE.std(0) / np.sqrt(len(ITEM10['seeds']))
fig, ax = plt.subplots(figsize=ITEM10['figsize'])
ax.plot(sizes, mean, '-o', color=COLORS['data']['generative'], label='generative basis')
ax.fill_between(sizes, mean - sem, mean + sem, color=COLORS['data']['generative'], alpha=0.3)
ax.axhline(ve_real, ls='--', color=COLORS['data']['real'], label='real tuning curves')
ax.set_xscale('log'); ax.set_xlabel('# sampled curves per variable')
ax.set_ylabel(f'variance explained'); ax.legend()
ax.set_title(f'Variance explained by sampled tuning curves (animal: {EX_ANIMAL_NAME}, session: {EX_SESSION_NAME})')
fig.tight_layout(); finalize(fig, 'item10_generative_ve'); del d

saved presentation_figures/item10_generative_ve.png


## 11. 1-D tuning split by barrier_flipped (selected cells)

In [56]:
ITEM11 = dict(
    cells=ITEM2['cells'] if 'ITEM2' in dir() else
          {'hdir': [108, 54, 2, 64], 'bdir': [79, 96, 108, 50],
           'hsa': [73, 108, 75, 58], 'hba': [71, 57, 115, 108]},
    figsize=(16, 13), lw=2.0,
    ylabel='Normalized firing rate',
    legend_labels={'false': 'barrier_flipped = False', 'true': 'barrier_flipped = True'},
)

In [57]:
from matplotlib.lines import Line2D

FbF, FbT = RES['barrier']['FbF'], RES['barrier']['FbT']
ncol = 4
fig = plt.figure(figsize=ITEM11['figsize'], layout='constrained')
# one subfigure per behavioural variable (bold label above the row) + a thin legend row
row_sfs = fig.subfigures(len(BEHAVE) + 1, 1, height_ratios=[1] * len(BEHAVE) + [0.18])
for p, v in enumerate(BEHAVE):
    sf = row_sfs[p]
    sf.suptitle(VAR_NAMES[v], fontsize=FONT['title'], fontweight='bold')
    axes = sf.subplots(1, ncol, sharex=True)
    for j, nidx in enumerate(ITEM11['cells'][v]):
        ax = axes[j]
        ax.plot(ANGLE, FbF[nidx, p], color=COLORS['data']['partition_false'], lw=ITEM11['lw'])
        ax.plot(ANGLE, FbT[nidx, p], color=COLORS['data']['partition_true'], lw=ITEM11['lw'])
        ax.set_title(f'#{nidx}', fontsize=FONT['title'])
        ax.set_xlim(-np.pi, np.pi)
        ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
        if j == 0:
            ax.set_ylabel(ITEM11['ylabel'])

# shared barrier_flipped legend below the whole figure
handles = [Line2D([], [], color=COLORS['data']['partition_false'], lw=ITEM11['lw'],
                  label=ITEM11['legend_labels']['false']),
           Line2D([], [], color=COLORS['data']['partition_true'], lw=ITEM11['lw'],
                  label=ITEM11['legend_labels']['true'])]
row_sfs[-1].legend(handles=handles, loc='center', ncol=2, frameon=False, fontsize=FONT['legend'])
fig.suptitle(f'1-D tuning split by barrier_flipped (animal: {EX_ANIMAL_NAME}, session: {EX_SESSION_NAME})')
finalize(fig, 'item11_barrier_split')

saved presentation_figures/item11_barrier_split.png


## 12. Tuning COM correlation: before vs after the barrier flip

Circular centre of mass of each cell's 1-D tuning curve in the
`barrier_flipped=False` (before) and `barrier_flipped=True` (after) partitions,
plotted against each other per variable with the circular correlation.

In [58]:
ITEM12 = dict(restrict_to_filtered=True, point_size=16, alpha=0.6, figsize=(16, 12))

In [59]:
FbF, FbT = RES['barrier']['FbF'], RES['barrier']['FbT']     # (N,P,D1) before / after
dom = [ANGLE] * len(BEHAVE)
comF = L.circular_com(FbF, dom)                              # (N,P) before flip
comT = L.circular_com(FbT, dom)                              # (N,P) after flip
if ITEM12['restrict_to_filtered']:
    m = L.significance_mask(RES['fit1d'][(EX_ANIMAL, EX_SESSION)]['scores'], RES['T'])
    comF, comT = comF[m], comT[m]

fig, axes = plt.subplots(2, 2, figsize=ITEM12['figsize'])
axes = axes.flatten()
for p, v in enumerate(BEHAVE):
    ax = axes[p]; rc = L.circular_corr(comF[:, p], comT[:, p])
    ax.scatter(comF[:, p], comT[:, p], s=ITEM12['point_size'],
               alpha=ITEM12['alpha'], color=COLORS['behave'][v])
    ax.plot([-np.pi, np.pi], [-np.pi, np.pi], 'k--', lw=1)
    ax.set_xlim(-np.pi, np.pi); ax.set_ylim(-np.pi, np.pi); ax.set_aspect('equal')
    ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
    ax.set_yticks([-np.pi, 0, np.pi]); ax.set_yticklabels([r'$-\pi$', '0', r'$\pi$'])
    ax.set_title(f'{VAR_NAMES[v]}\n$r_c$ = {rc:.2f}')
    ax.set_xlabel('COM before flip')
    if p == 0:
        ax.set_ylabel('COM after flip')
    print(f"{v}: circular corr = {rc:.3f}")
n_used = comF.shape[0]
fig.suptitle(f'1-D Tuning COM before vs after barrier flip '
             f'(animal: {EX_ANIMAL_NAME}, session: {EX_SESSION_NAME}, N={n_used} cells)')
fig.tight_layout(); finalize(fig, 'item12_barrier_com_corr')

hdir: circular corr = 0.889
bdir: circular corr = -0.355
hsa: circular corr = 0.858
hba: circular corr = 0.138
saved presentation_figures/item12_barrier_com_corr.png
